# 🛡️ Open Fraud Intelligence — Jupyter Notebook
## Analiză completă a dataset-ului v2

Acest notebook demonstrează:
1. Încărcarea și explorarea dataset-ului
2. Statistici descriptive
3. Clasificare simplă cu Scikit-learn
4. Scam DNA analysis
5. Vizualizări cu Matplotlib/Seaborn
6. Export pentru fine-tuning LLM

In [ ]:
# Instalare dependențe
# !pip install pandas matplotlib seaborn scikit-learn transformers

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import Counter

# Stil dark pentru grafice
plt.style.use('dark_background')
ACCENT = '#6366f1'
RED    = '#ef4444'
GREEN  = '#22c55e'
print('✅ Librării importate')

In [ ]:
# ── 1. Încărcare dataset ──────────────────────────────────────────────────
dataset_path = Path('../../datasets/scams_v2.json')

with open(dataset_path, encoding='utf-8') as f:
    data = json.load(f)

print(f'📊 Dataset încărcat: {len(data)} intrări')

# Convertire în DataFrame
rows = []
for entry in data:
    ai = entry.get('ai_dataset', {})
    ver = entry.get('verification', {})
    conf_obj = entry.get('confidence', {})
    rows.append({
        'id':            entry.get('id', ''),
        'title':         entry.get('title', ''),
        'platform':      entry.get('platform', ''),
        'severity':      entry.get('severity', ''),
        'year':          entry.get('year'),
        'active':        entry.get('active'),
        'country':       entry.get('country', ''),
        'tags':          ','.join(entry.get('tags', [])),
        'ai_label':      ai.get('label', ''),
        'ai_confidence': ai.get('confidence', 0.0),
        'ai_message':    ai.get('message', ''),
        'msg_length':    len(ai.get('message', '')),
        'msg_words':     len(ai.get('message', '').split()),
        'red_flags_count': len(entry.get('red_flags', [])),
        'has_ioc':       bool(any(entry.get('ioc', {}).get(k) for k in ['domains','urls','phones'])),
        'has_mitre':     bool(entry.get('mitre_attack')),
        'ver_status':    ver.get('status', 'pending'),
        'overall_confidence': conf_obj.get('score', 0.0)
    })

df = pd.DataFrame(rows)
print(f'\n📋 Coloane: {list(df.columns)}')
df.head(3)

In [ ]:
# ── 2. Statistici descriptive ─────────────────────────────────────────────
print('=== STATISTICI GENERALE ===')
print(f'Total intrări:      {len(df)}')
print(f'Fraude scam:        {(df.ai_label=="scam").sum()}')
print(f'Intrări legitimate: {(df.ai_label=="legitimate").sum()}')
print(f'Cu IOC:             {df.has_ioc.sum()}')
print(f'Cu MITRE ATT&CK:    {df.has_mitre.sum()}')
print(f'\nDistribuție platforme:')
print(df.platform.value_counts().to_string())
print(f'\nDistribuție severitate (scam):')
print(df[df.ai_label=="scam"].severity.value_counts().to_string())
print(f'\nConfidence AI — statistici:')
print(df.ai_confidence.describe())

In [ ]:
# ── 3. Vizualizări ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.patch.set_facecolor('#0f1117')
for ax in axes.flat:
    ax.set_facecolor('#1a1d27')
    ax.spines['bottom'].set_color('#2d3252')
    ax.spines['left'].set_color('#2d3252')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(colors='#94a3b8')

# 3.1 Distribuție platforme
ax = axes[0,0]
plat = df.platform.value_counts().head(8)
colors = ['#6366f1','#8b5cf6','#06b6d4','#22c55e','#f97316','#ef4444','#eab308','#ec4899']
bars = ax.barh(plat.index, plat.values, color=colors[:len(plat)])
ax.set_title('Distribuție platforme', color='#e2e8f0', fontweight='bold')
ax.set_xlabel('Număr intrări', color='#94a3b8')
for bar, val in zip(bars, plat.values):
    ax.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2, str(val), va='center', color='#e2e8f0', fontsize=10)

# 3.2 Label distribution donut
ax = axes[0,1]
label_counts = df.ai_label.value_counts()
ax.pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%',
       colors=[RED, GREEN], startangle=90, textprops={'color':'#e2e8f0'},
       wedgeprops={'edgecolor':'#0f1117', 'linewidth':2})
ax.set_title('Distribuție AI Labels', color='#e2e8f0', fontweight='bold')

# 3.3 Confidence histogram
ax = axes[0,2]
scam_conf  = df[df.ai_label=="scam"].ai_confidence
legit_conf = df[df.ai_label=="legitimate"].ai_confidence
ax.hist(scam_conf,  bins=15, alpha=0.7, color=RED,   label='Scam',      edgecolor='#0f1117')
ax.hist(legit_conf, bins=15, alpha=0.7, color=GREEN, label='Legitimate', edgecolor='#0f1117')
ax.set_title('Distribuție Confidence AI', color='#e2e8f0', fontweight='bold')
ax.set_xlabel('Confidence Score', color='#94a3b8')
ax.legend(facecolor='#21253a', labelcolor='#e2e8f0')

# 3.4 Severitate (scam)
ax = axes[1,0]
sev_order = ['Critical','High','Medium','Low','Informational']
sev_colors = {'Critical':'#7c0000','High':'#f97316','Medium':'#eab308','Low':'#22c55e','Informational':'#6b7280'}
sev_counts = df[df.ai_label=="scam"].severity.value_counts()
sev_vals = [sev_counts.get(s,0) for s in sev_order]
sev_cols = [sev_colors[s] for s in sev_order]
ax.bar(sev_order, sev_vals, color=sev_cols, edgecolor='#0f1117')
ax.set_title('Severitate fraude', color='#e2e8f0', fontweight='bold')
ax.set_ylabel('Număr', color='#94a3b8')

# 3.5 Lungime mesaje
ax = axes[1,1]
scam_len  = df[df.ai_label=="scam"].msg_words
legit_len = df[df.ai_label=="legitimate"].msg_words
bp = ax.boxplot([scam_len, legit_len], labels=['Scam','Legitimate'],
               patch_artist=True, medianprops={'color':'white'})
bp['boxes'][0].set_facecolor('#ef444466')
bp['boxes'][1].set_facecolor('#22c55e66')
ax.set_title('Număr cuvinte mesaje', color='#e2e8f0', fontweight='bold')
ax.set_ylabel('Cuvinte', color='#94a3b8')

# 3.6 Red flags count
ax = axes[1,2]
rf_scam  = df[df.ai_label=="scam"].red_flags_count
ax.hist(rf_scam, bins=8, color=ACCENT, edgecolor='#0f1117', alpha=0.8)
ax.set_title('Distribuție Red Flags per intrare', color='#e2e8f0', fontweight='bold')
ax.set_xlabel('Număr red flags', color='#94a3b8')
ax.set_ylabel('Frecvență', color='#94a3b8')

plt.tight_layout(pad=3)
plt.savefig('ofi_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('✅ Grafic salvat: ofi_analysis.png')

In [ ]:
# ── 4. Clasificare simplă cu Scikit-learn ─────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Filtrare: scam + legitimate
df_ml = df[df.ai_label.isin(['scam','legitimate'])].copy()
df_ml['label_binary'] = (df_ml.ai_label == 'scam').astype(int)

print(f'Samples pentru ML: {len(df_ml)} (scam={df_ml.label_binary.sum()}, legit={len(df_ml)-df_ml.label_binary.sum()})')

if len(df_ml) >= 10:
    X = df_ml.ai_message
    y = df_ml.label_binary

    # TF-IDF + Logistic Regression
    vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1,2), min_df=1)
    X_vec = vectorizer.fit_transform(X)

    # Cross-validation
    clf = LogisticRegression(random_state=42, max_iter=1000)
    scores = cross_val_score(clf, X_vec, y, cv=min(5, len(df_ml)//2), scoring='f1')
    print(f'\n📊 Cross-validation F1 scores: {scores.round(3)}')
    print(f'Medie F1: {scores.mean():.3f} (+/- {scores.std():.3f})')

    # Antrenare pe tot dataset-ul
    clf.fit(X_vec, y)
    
    # Test pe câteva mesaje noi
    test_messages = [
        'Am plătit deja. Accesați urgent linkul pentru confirmare.',
        'Bună ziua, mai este disponibil produsul?',
        'Profit garantat 300% lunar din crypto! Investiție fără risc!',
        'Coletul nu a putut fi livrat. Plătiți taxa: [link]',
    ]
    X_test = vectorizer.transform(test_messages)
    preds = clf.predict(X_test)
    probs = clf.predict_proba(X_test)[:,1]
    
    print('\n🔍 Predicții mesaje noi:')
    for msg, pred, prob in zip(test_messages, preds, probs):
        label = '🔴 SCAM' if pred == 1 else '🟢 LEGIT'
        print(f'  {label} ({prob:.2%}) — {msg[:60]}...')
else:
    print('⚠️ Prea puține sample-uri pentru antrenare ML. Adaugă mai multe intrări în dataset.')

In [ ]:
# ── 5. Export Fine-Tuning Dataset (format JSONL pentru LLM) ───────────────
import jsonlines  # pip install jsonlines

output_path = Path('ofi_finetune.jsonl')
count = 0

with open(output_path, 'w', encoding='utf-8') as f:
    for entry in data:
        ai = entry.get('ai_dataset', {})
        if not ai.get('message') or not ai.get('label'):
            continue
        
        # Format prompt pentru instruction tuning
        prompt = f"""Analizează dacă următorul mesaj este o tentativă de fraudă sau un mesaj legitim.

Mesaj: {ai['message']}

Platformă: {entry.get('platform', 'necunoscută')}
"""
        completion = f"""Clasificare: {'FRAUDĂ (SCAM)' if ai['label'] == 'scam' else 'MESAJ LEGITIM'}
Confidence: {ai.get('confidence', 0):.0%}
Explicație: {entry.get('scenario', 'N/A')[:200]}
"""
        record = {
            'id':         entry.get('id'),
            'prompt':     prompt,
            'completion': completion,
            'label':      ai['label'],
            'platform':   entry.get('platform')
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        count += 1

print(f'✅ Fine-tuning dataset exportat: {output_path} ({count} exemple)')
print(f'   Format: JSONL — compatibil cu OpenAI fine-tuning, Hugging Face, Axolotl')